# switchgear-estimation-assistant: retrieval on Colab

Independent personal project. Synthetic data only.

**About the saved outputs.** They were recorded before the restructure, on the code at commit `d4e93d5` or earlier, and some come from older versions of the code. For example, an output may show a plain-text result or list only three tools. This notebook has not yet been re-run since the restructure. Imports and paths are updated, and new cells have no output.


In [1]:
# Cell 1: Check whether the server and model survived
!ollama list

/bin/bash: line 1: ollama: command not found


In [2]:
# Cell 2: Confirm Colab gave us a GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [3]:
# Cell 3: Install zstd (unpacking) and pciutils + lshw (GPU detection)
!apt-get install -y -q zstd pciutils lshw

Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libpci3 pci.ids usb.ids
The following NEW packages will be installed:
  libpci3 lshw pci.ids pciutils usb.ids zstd
0 upgraded, 6 newly installed, 0 to remove and 33 not upgraded.
Need to get 1,583 kB of archives.
After this operation, 5,343 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 pci.ids all 0.0~2024.03.31-1ubuntu0.1 [275 kB]
Get:2 http://archive.ubuntu.com/ubuntu noble/main amd64 libpci3 amd64 1:3.10.0-2build1 [36.5 kB]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 lshw amd64 02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1 [334 kB]
Get:4 http://archive.ubuntu.com/ubuntu noble/main amd64 pciutils amd64 1:3.10.0-2build1 [69.7 kB]
Get:5 http://archive.ubuntu.com/ubuntu noble/main amd64 usb.ids all 2024.03.18-1 [223 kB]
Get:6 http://archive.ubuntu.com/ubuntu noble-update

In [4]:
# Cell 4: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
# Cell 5: Start the Ollama server in the background
!nohup ollama serve > ollama.log 2>&1 &
!sleep 8

In [6]:
# Cell 6: Install the Ollama Python client
!pip install -q ollama

In [7]:
# Cell 7: Download the text model
import ollama

print(ollama.pull("gemma4:e2b").status)

success


In [8]:
# Cell 8: Download the embedding model
import ollama

print(ollama.pull("nomic-embed-text").status)

success


In [9]:
# Cell 9: Turn one sentence into a vector

response = ollama.embed(
    model = "nomic-embed-text",
    input = "Cable glands seal the point where a cable enters an enclosure."
)

vector = response.embeddings[0]
print(len(vector))
print(vector[:5])

768
[0.030740838, 0.05348693, -0.14595363, -0.055521727, 0.02450215]


In [10]:
# Cell 10: Compare two sentences with cosine similarity

import math 
sentences = [
    "Cable glands seal the point where a cable enters an enclosure.",
    "How do I close the gap where a wire goes into a box?",
    "A busbar carries current between circuit breakers in a panel.",
]

# One call can embed several sentences; the vectors come back in the same order
vectors = ollama.embed(model = "nomic-embed-text", input = sentences).embeddings


def cosine (a, b):
    """Return the cosine similarity of two vectors (1 = same direction)."""
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(y * y for y in b))
    return dot / (length_a * length_b)

print("0 vs 1:", round(cosine(vectors[0], vectors[1]), 3))
print("0 vs 2:", round(cosine(vectors[0], vectors[2]), 3))
print("1 vs 2:", round(cosine(vectors[1], vectors[2]), 3))

0 vs 1: 0.574
0 vs 2: 0.495
1 vs 2: 0.532


In [ ]:
# Cell 11: Cosine similarity with prints, on three-number vectors
import math


def cosine_verbose(a, b):
    """Return the cosine similarity of two vectors, printing each step."""
    print("a =", a)
    print("b =", b)

    # Pair the numbers: first with first, second with second, and so on
    pairs = list(zip(a, b))
    print("pairs (x, y) =", pairs)

    # Multiply each pair, then add the products
    products = [x * y for x, y in pairs]
    dot = sum(products)
    print("products x*y =", products)
    print("dot =", dot)

    # The size of each vector: square every number, add, take the square root
    squares_a = [x * x for x in a]
    squares_b = [y * y for y in b]
    length_a = math.sqrt(sum(squares_a))
    length_b = math.sqrt(sum(squares_b))
    print("squares of a =", squares_a, "-> length_a =", round(length_a, 3))
    print("squares of b =", squares_b, "-> length_b =", round(length_b, 3))

    # Divide by both sizes so only the direction counts
    score = dot / (length_a * length_b)
    print("score = dot / (length_a * length_b) =", round(score, 3))
    return score


cosine_verbose([1, 2, 3], [3, 2, 1])

a = [1, 2, 3]
b = [3, 2, 1]
pairs (x, y) = [(1, 3), (2, 2), (3, 1)]
products x*y = [3, 4, 3]
dot = 10
squares of a = [1, 4, 9] -> length_a = 3.742
squares of b = [9, 4, 1] -> length_b = 3.742
score = dot / (length_a * length_b) = 0.714


0.7142857142857143

In [12]:
# Cell 12: Rank four notes for one question
# Four invented notes. The ids are the sources we will show in answers later.

notes = {
    "D1": "A moulded case circuit breaker switches off a circuit when the current is too high.",
    "D2": "A busbar carries current between circuit breakers in a panel.",
    "D3": "Cable glands seal the point where a cable enters an enclosure.",
    "D4": "A current transformer measures the current in a cable so that a meter can read it.",
}

# Embed all four notes in one call; the vectors keep the same order as the notes
note_vectors = ollama.embed(
    model = "nomic-embed-text",
    input = ["search_document: " + text for text in notes.values()]
).embeddings

# Embed the question in the same way
question = " How do I close the gap where a wire goes into a box?"
question_vector = ollama.embed(
    model = "nomic-embed-text",
    input = "search_query: " + question
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores = []
for note_id, note_vector in zip(notes.keys(), note_vectors):
    scores.append((cosine(question_vector, note_vector), note_id))

# Highest score first (sort orders pairs by their first item, the score)
scores.sort(reverse=True)

for score, note_id in scores:
    print(f"{note_id}: {round(score, 3)}")

D2: 0.583
D1: 0.582
D3: 0.581
D4: 0.512


In [13]:
# Cell 12b: Rank four notes for one question
# Four invented notes. The ids are the sources we will show in answers later.

notes = {
    "D1": "A moulded case circuit breaker switches off a circuit when the current is too high.",
    "D2": "A busbar carries current between circuit breakers in a panel.",
    "D3": "Cable glands seal the point where a cable enters an enclosure.",
    "D4": "A current transformer measures the current in a cable so that a meter can read it.",
}

# Embed all four notes in one call; the vectors keep the same order as the notes
note_vectors = ollama.embed(
    model = "nomic-embed-text",
    input = ["search_document: " + text for text in notes.values()]
).embeddings

# Embed the question in the same way
question = "Which part carries current between breakers in a panel?" # The question is differnt in this version.
question_vector = ollama.embed(
    model = "nomic-embed-text",
    input = "search_query: " + question
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores = []
for note_id, note_vector in zip(notes.keys(), note_vectors):
    scores.append((cosine(question_vector, note_vector), note_id))

# Highest score first (sort orders pairs by their first item, the score)
scores.sort(reverse=True)

for score, note_id in scores:
    print(f"{note_id}: {round(score, 3)}")

D2: 0.833
D1: 0.67
D4: 0.635
D3: 0.524


This is a clear win. D2 scores 0.833, and the next note is 0.16 behind. So the model works when the question shares words with a note. It struggles when the question uses different words, as in the “wire in a box” question.

Question	Winner	Gap to second place
Easy: “Which part carries current between breakers in a panel?”	D2 (correct)	0.163
Hard: “How do I close the gap where a wire goes into a box?”	D2 (wrong)	0.002

Now we test a stronger embedding model on the same two questions in **cell 15**

The call returns one reply object. It holds the vectors and some extra details. .embeddings picks out the vectors and drops the rest. These two versions do the same job:

```py

# Version A: keep the whole reply, then pick the vectors out of it
response = ollama.embed(model="nomic-embed-text", input=list(notes.values()))
note_vectors = response.embeddings

# Version B: pick them out straight away (what Cell 12 does)
note_vectors = ollama.embed(model="nomic-embed-text", input=list(notes.values())).embeddings
```

```
response                 the whole reply
├── model                the model name
├── ...                  other details
└── embeddings           the list of vectors  ← this is note_vectors
    ├── [0]              vector for D1 (768 numbers)
    ├── [1]              vector for D2
    ├── [2]              vector for D3
    └── [3]              vector for D4

```
An everyday picture: the reply is a parcel. .embeddings takes out the one item you want. You throw the box away.

In [14]:
# Cell 13: Compare the whole reply with its embeddings part
response = ollama.embed(model="nomic-embed-text", input=list(notes.values()))

print(type(response))
print(type(response.embeddings))
print(len(response.embeddings))
print(len(response.embeddings[0]))

print("first five numbers of note D1:", response.embeddings[0][:5])

<class 'ollama._types.EmbedResponse'>
<class 'list'>
4
768
first five numbers of note D1: [0.067672305, -0.0007982441, -0.18156846, -0.0222183, 0.04234001]


response.embeddings[0] is the note D1, the whole sentence: “A moulded case circuit breaker switches off a circuit when the current is too high.”
The 768 numbers stand for the meaning of that entire sentence, not for one word in it.
* D1 (13 words)  ->  response.embeddings[0]  ->  768 numbers
* D2 (11 words)  ->  response.embeddings[1]  ->  768 numbers

What happens inside: the embedding model reads all the words together, including their order and context. It then gives back one list of 768 numbers for the whole text. There are no per-word numbers in the output.

An everyday picture: think of the 768 numbers as an address on a map of meaning. A short address would not be enough to tell thousands of meanings apart, so the address has 768 parts. Every text gets one address. Texts with similar meaning get addresses close together.

Two facts follow:

* Every note gets exactly 768 numbers, long or short.
* The individual numbers mean nothing alone, such as 0.0677 or -0.0008. Only the whole address, compared with other addresses, carries meaning.

In [15]:
# Cell 14: Check that any length of text gives one vector of the same size
short = ollama.embed(model="nomic-embed-text", input="busbar").embeddings
long = ollama.embed(model="nomic-embed-text", input=notes["D1"] + " " + notes["D2"]).embeddings

print("one word:     ", len(short), "vector(s) of", len(short[0]), "numbers")
print("two sentences:", len(long), "vector(s) of", len(long[0]), "numbers")

one word:      1 vector(s) of 768 numbers
two sentences: 1 vector(s) of 768 numbers


The size is built into the model when it is trained.
```
Embedding model	Numbers per vector
all-MiniLM-L6-v2	384
nomic-embed-text (yours)	768
mxbai-embed-large	1024
OpenAI text-embedding-3-small	1536
OpenAI text-embedding-3-large	3072
```

In [16]:
# Cell 15: Download a larger embedding model (1024 numbers per vector)
import ollama

print(ollama.pull("mxbai-embed-large").status)

success


In [17]:
# Cell 16a: Rank the same four notes with the larger embedding model

# All four note vectors (no [0]), embedded by the larger model
note_vectors_large = ollama.embed(
    model="mxbai-embed-large",  # larger model, 1024 numbers per vector
    input=list(notes.values()),
).embeddings

# One question, one vector ([0] takes it out of the list of one)
question = "How do I close the gap where a wire goes into a box?"
question_vector_large = ollama.embed(
    model="mxbai-embed-large",
    input=question,
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores_large = []
for note_id, note_vector in zip(notes.keys(), note_vectors_large):
    scores_large.append((cosine(question_vector_large, note_vector), note_id))

scores_large.sort(reverse=True)

for score, note_id in scores_large:
    print(f"{note_id}: {round(score, 3)}")

D3: 0.699
D2: 0.577
D1: 0.544
D4: 0.534


In [18]:
# Cell 16b: Rank the same four notes with the larger embedding model

# All four note vectors (no [0]), embedded by the larger model
note_vectors_large = ollama.embed(
    model="mxbai-embed-large",  # larger model, 1024 numbers per vector
    input=list(notes.values()),
).embeddings

# One question, one vector ([0] takes it out of the list of one)
question = "Which part carries current between breakers in a panel?"
question_vector_large = ollama.embed(
    model="mxbai-embed-large",
    input=question,
).embeddings[0]

# Score the question against every note; each pair is (score, note id)
scores_large = []
for note_id, note_vector in zip(notes.keys(), note_vectors_large):
    scores_large.append((cosine(question_vector_large, note_vector), note_id))

scores_large.sort(reverse=True)

for score, note_id in scores_large:
    print(f"{note_id}: {round(score, 3)}")

D2: 0.858
D1: 0.703
D4: 0.575
D3: 0.572


Retrieval found the right note. Now the model writes the answer from that note only. The source comes from our code, not from the model. We know which note we handed over, so the source cannot be made up.

In [19]:
# Cell 17: Answer from the best note and show its source

# The top pair is (score, note id)

best_score, best_id = scores_large[0]
best_note = notes[best_id]

response = ollama.chat(
    model = "gemma4:e2b",
    messages = [
        {
            "role": "system",
            "content": ("Answer the question using the note provided. "
                        "If the note does not contain the answer, say 'I don't know.'")
        },
        {
            "role": "user",
            "content": f"Note:\n{best_note}\n\nQuestion:\n{question}"
        },
    ],
    think = False,
)
print(response.message.content)
print(f"Source: {best_id} (score {round(best_score, 3)})")

A busbar carries current between circuit breakers in a panel.
Source: D2 (score 0.858)


**The next cell (Cell 18) has no proof.** Its saved run failed with `NameError: name 'notes' is not defined`. The cause was not recorded. The output was cleared. The results typed up in `tutorials/03-rag-basics.ipynb` (steps 4 and 5) cannot be checked here.


In [20]:
# Cell 18: Ask several questions in a loop

def answer_from_notes(question):
    """Find the best note for a question and answer from it.
    Returns the answer text, the id of the note used (None if refused), and the best score."""
    # Embed the question with the same model used for the notes
    question_vector = ollama.embed(
        model="mxbai-embed-large",
        input=question,
    ).embeddings[0]

    # Score every note; each pair is (score, note id)
    scores = []
    for note_id, note_vector in zip(notes.keys(), note_vectors_large):
        scores.append((cosine(question_vector, note_vector), note_id))
    scores.sort(reverse=True)
    best_score, best_id = scores[0]

    # Answer from the best note only
    response = ollama.chat(
        model="gemma4:e2b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Answer the question using only the note provided. "
                    "If the note does not contain the answer, say: I cannot answer from the notes."
                ),
            },
            {
                "role": "user",
                "content": f"Note:\n{notes[best_id]}\n\nQuestion: {question}",
            },
        ],
        think=False,
    )

    answer = response.message.content
    # A refusal means the note did not support an answer, so no source is reported
    refused = "cannot answer" in answer.lower()
    source = None if refused else best_id
    return answer, source, best_score


questions = [
    "Which part carries current between breakers in a panel?",
    "How do I close the gap where a wire goes into a box?",
    "What is the price of a busbar?",
    "What lets a meter read the current in a cable?",
]

for q in questions:
    answer, source, score = answer_from_notes(q)
    print("Q:", q)
    print("A:", answer)
    if source is None:
        print(f"Source: none (best note scored {round(score, 3)}, but it did not hold the answer)")
    else:
        print(f"Source: {source} (score {round(score, 3)})")
    print()

Q: Which part carries current between breakers in a panel?
A: A busbar carries current between circuit breakers in a panel.
Source: D2 (score 0.858)

Q: How do I close the gap where a wire goes into a box?
A: Cable glands seal the point where a cable enters an enclosure.
Source: D3 (score 0.699)

Q: What is the price of a busbar?
A: I cannot answer from the notes.
Source: none (best note scored 0.609, but it did not hold the answer)

Q: What lets a meter read the current in a cable?
A: A current transformer measures the current in a cable so that a meter can read it.
Source: D4 (score 0.913)



In [21]:
# Cell 24: Download the embedding model used by retrieval.py
import ollama

print(ollama.pull("mxbai-embed-large").status)

success


In [22]:
# Cell 24b: Create the package folder and let Python find it
!mkdir -p ai_engine/estimation_assistant
!touch ai_engine/estimation_assistant/__init__.py

import sys

sys.path.insert(0, "ai_engine")


### Cell 24c: Write schemas.py to the session


In [23]:
%%writefile ai_engine/estimation_assistant/schemas.py
from pydantic import BaseModel

class ToolStep(BaseModel):
    name: str
    arguments: dict
    result: str

class AgentResult(BaseModel):
    answer: str
    steps: list[ToolStep]
    sources: list[str] = []
    unsupported: list[float] = []



Writing src/estimation_assistant/schemas.py


### Cell 24d: Write tools.py to the session


In [24]:
%%writefile ai_engine/estimation_assistant/tools.py
"""Tools for interacting with the product catalog (synthetic data)."""

PRICES = {
    "mccb": 145.00,
    "acb": 2850.00,
    "busbar": 95.00,
    "current transformer": 38.00,
    "cable gland": 25.00,
}


def lookup_price(part_name: str) -> str:
    """Return the unit price of one part, as text (for example 95.0).
    Valid part names: mccb (moulded case circuit breaker),
    acb (air circuit breaker), busbar (copper busbar, per metre),
    current transformer, cable gland.
    Use the exact name in lowercase.
    If the name is not valid, the reply says so and lists the valid part names.
    """
    #Normalize the part name before look up
    name = part_name.strip().lower()
    if name in PRICES:
        return str(PRICES[name])
    return f"unknown part '{part_name}'. Valid part: {', '.join(PRICES)}"


def list_parts() -> str:
    """List every part in the catalog, separated by commas."""
    return ", ".join(PRICES)

def line_total(unit_price: float, quantity: int) -> float:
    """Return the total price for one quote line, rounded to 2 decimal places.
    unit_price is the price of one item in pounds. Use the value from lookup_price.
    quantity is the number of items, as a whole number.
    """
    # Round to 2 decimals so the amount is a clean currency value
    return round(unit_price * quantity, 2)

Writing src/estimation_assistant/tools.py


### Cell 24e: Write retrieval.py to the session


In [25]:
%%writefile ai_engine/estimation_assistant/retrieval.py
"""Searches a small set of synthetic technical notes by meaning (embeddings)."""

import math
import ollama

EMBED_MODEL = "mxbai-embed-large"

NOTES = {
    "D1": "A moulded case circuit breaker switches off a circuit when the current is too high.",
    "D2": "A busbar carries current between circuit breakers in a panel.",
    "D3": "Cable glands seal the point where a cable enters an enclosure.",
    "D4": "A current transformer measures the current in a cable so that a meter can read it.",
}

def cosine_notes(a: list[float], b: list[float]) -> float:
    """Return the cosine similarity of two vectors (1 = same direction)."""
    dot = sum(x * y for x, y in zip(a, b))
    length_a = math.sqrt(sum(x * x for x in a))
    length_b = math.sqrt(sum(y * y for y in b))
    return dot / (length_a * length_b)

def rank_notes(question: str) ->list[tuple[float, str]]:
    """Return (score, note id) pairs for a question, best match first."""

    # Notes and question must be embedded by the same model
    note_vectors = ollama.embed(
        model=EMBED_MODEL,
        input=list(NOTES.values())
    ).embeddings

    question_vector = ollama.embed(
        model=EMBED_MODEL,
        input = question
    ).embeddings[0]

    scores = []
    for note_id, note_vector in zip(NOTES, note_vectors):
        score = cosine_notes(note_vector, question_vector)
        scores.append((score, note_id))
    scores.sort(reverse=True)
    return scores


def search_notes(question: str) -> str:
    """Find the technical note that best matches a question, by meaning.
    Use it for questions about what a part does or how it is used.
    Do not use it for prices; use lookup_price for those.
    The reply gives the note id (the source), the match score and the note text.
    """
    best_score, best_id = rank_notes(question)[0]
    return f"Source {best_id} (score {round(best_score, 3)}): {NOTES[best_id]}"


Writing src/estimation_assistant/retrieval.py


### Cell 24f: Write checks.py to the session


In [26]:
%%writefile ai_engine/estimation_assistant/checks.py
"""Checks that an agent answer is backed by its tool results."""

import re
from .schemas import AgentResult

# A number: digits with an optional thousands separator and decimals.
# The lookbehind skips digits that belong to a word, such as the 2 in "D2".
NUMBER = re.compile(
    r"(?<![A-Za-z0-9])(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)"
)


def find_numbers(text: str) -> list[float]:
    """Return every number in the text as a float."""
    numbers = []
    for match in NUMBER.findall(text):
        # Remove thousands separators so "2,850.00" becomes 2850.0
        numbers.append(float(match.replace(",", "")))
    return numbers

def unsupported_numbers(question: str, result: AgentResult) -> list[float]:
    """Return the numbers in the answer that appear nowhere in the question or the run log."""
    # Collect every number the answer is allowed to use, rounded to 2 decimals
    allowed = set()
    for number in find_numbers(question):
        allowed.add(round(number, 2))
    for step in result.steps:
        for number in find_numbers(step.result) + find_numbers(str(step.arguments)):
            allowed.add(round(number, 2))

    # Anything in the answer outside that set was not backed by evidence
    return [n for n in find_numbers(result.answer) if round(n, 2) not in allowed]

Writing src/estimation_assistant/checks.py


### Cell 24g: Write agent.py to the session


In [27]:
%%writefile ai_engine/estimation_assistant/agent.py
"""Runs the tools from tools.py safely and drives the agent loop."""

import ollama

from .checks import unsupported_numbers
from .retrieval import search_notes
from .schemas import AgentResult, ToolStep
from .tools import line_total, list_parts, lookup_price

MODEL = "gemma4:e2b"

TOOLS = {
    "lookup_price": lookup_price,
    "list_parts": list_parts,
    "line_total": line_total,
    "search_notes": search_notes,
}


def run_tool(name: str, arguments: dict) -> str:
    """Run one tool by name and return its result as text.
    An unknown tool name, wrong inputs or any other error inside the tool return a message instead of raising an error."""
    # Reject names the model made up
    if name not in TOOLS:
        return f"Unknown tool '{name}'. Valid tools: {', '.join(TOOLS)}"
    try:
        return str(TOOLS[name](**arguments))
    except TypeError as error:
        # Raised when the inputs do not match the tool's parameters
        return f"Bad inputs for '{name}': {error}"
    except Exception as error:
        # Any other failure inside the tool is reported to the model instead of stopping the run
        return f"Tool '{name}' failed with {type(error).__name__}: {error}"

def find_sources(steps: list[ToolStep]) -> list[str]:
    """Return the note ids that search_notes returned during a run."""
    sources = []
    for step in steps:
        # A search result starts with "Source D2 (score ...): ..."
        if step.name == "search_notes" and step.result.startswith("Source "):
            sources.append(step.result.split()[1])
    return sources

def build_result(question: str, answer: str, steps: list[ToolStep]) -> AgentResult:
    """Build the run result, with its sources and any numbers the evidence does not back."""
    result = AgentResult(answer=answer, steps=steps, sources=find_sources(steps))
    result.unsupported = unsupported_numbers(question, result)
    return result

def run_agent(question: str, max_rounds: int = 5) -> AgentResult:
    """Ask the model, run the tools it requests, and return the answer with the steps taken."""
    # The history starts with the user's question
    messages = [{"role": "user", "content": question}]
    # A record of every tool run, kept for the final result
    steps: list[ToolStep] = []

    for round_number in range(1, max_rounds + 1):
        print(f"Round {round_number}")

        # Ask the model, sending the whole history and the tool functions
        response = ollama.chat(
            model=MODEL,
            messages=messages,
            tools=list(TOOLS.values()),
            think=False,
        )

        # Keep the model's reply in the history
        messages.append(response.message)

        # No tool request means the model has finished
        if not response.message.tool_calls:
            return build_result(question, response.message.content or "", steps)

        # Run each requested tool, record the step, and keep the result in the history
        for call in response.message.tool_calls:
            name = call.function.name
            arguments = dict(call.function.arguments)
            result = run_tool(name, arguments)
            print(f"  {name}({arguments}) -> {result}")
            steps.append(ToolStep(name=name, arguments=arguments, result=result))
            messages.append({"role": "tool", "tool_name": name, "content": result})

    return build_result(question, "Stopped: reached the round limit.", steps)

Writing src/estimation_assistant/agent.py


In [28]:
# Cell 26: Check that retrieval.py ranks notes and returns a source
import importlib

from estimation_assistant import retrieval

importlib.reload(retrieval)

print(retrieval.cosine_notes([1, 2, 3], [3, 2, 1]))

for score, note_id in retrieval.rank_notes("How do I close the gap where a wire goes into a box?"):
    print(note_id, round(score, 3))

print(retrieval.search_notes("Which part carries current between breakers in a panel?"))

0.7142857142857143
D3 0.699
D2 0.577
D1 0.544
D4 0.534
Source D2 (score 0.858): A busbar carries current between circuit breakers in a panel.


In [29]:
# Cell 27b: Confirm agent.py on Colab has the notes tool
!ls -l ai_engine/estimation_assistant/tools.py ai_engine/estimation_assistant/schemas.py ai_engine/estimation_assistant/retrieval.py ai_engine/estimation_assistant/agent.py
!grep -n "search_notes" ai_engine/estimation_assistant/agent.py

-rw-r--r-- 1 root root 3221 Nov  3 14:34 src/estimation_assistant/agent.py
-rw-r--r-- 1 root root 1909 Nov  3 14:34 src/estimation_assistant/retrieval.py
-rw-r--r-- 1 root root  245 Nov  3 14:34 src/estimation_assistant/schemas.py
-rw-r--r-- 1 root root 1333 Nov  3 14:34 src/estimation_assistant/tools.py
6:from .retrieval import search_notes
16:    "search_notes": search_notes,
33:    """Return the note ids that search_notes returned during a run."""
37:        if step.name == "search_notes" and step.result.startswith("Source "):


In [30]:
# Cell 28: Load the new agent.py and ask a question that needs the notes
import importlib
import sys

for name in ["estimation_assistant.agent"]:
    sys.modules.pop(name, None)
agent = importlib.import_module("estimation_assistant.agent")

result = agent.run_agent("Which part carries current between breakers in a panel?")
print(result.answer)
for step in result.steps:
    print(step.name, step.arguments, "->", step.result)

Round 1
  search_notes({'question': 'which part carries current between breakers in a panel'}) -> Source D2 (score 0.861): A busbar carries current between circuit breakers in a panel.
Round 2
The part that carries current between breakers in a panel is a **busbar**.
search_notes {'question': 'which part carries current between breakers in a panel'} -> Source D2 (score 0.861): A busbar carries current between circuit breakers in a panel.


In [31]:
# Cell 29: Check that a price question still uses the price tool
result = agent.run_agent("What is the price of one busbar?")
print(result.answer)
for step in result.steps:
    print(step.name, step.arguments, "->", step.result)

Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
The price of one busbar is 95.0.
lookup_price {'part_name': 'busbar'} -> 95.0


In [32]:
# Cell 30: Check that the result carries its sources
import importlib
import sys

for name in ["estimation_assistant.agent", "estimation_assistant.schemas"]:
    sys.modules.pop(name, None)
agent = importlib.import_module("estimation_assistant.agent")

notes_result = agent.run_agent("Which part carries current between breakers in a panel?")
print("answer: ", notes_result.answer)
print("sources:", notes_result.sources)

price_result = agent.run_agent("What is the price of one busbar?")
print("answer: ", price_result.answer)
print("sources:", price_result.sources)

Round 1
  search_notes({'question': 'which part carries current between breakers in a panel'}) -> Source D2 (score 0.861): A busbar carries current between circuit breakers in a panel.
Round 2
answer:  The part that carries current between breakers in a panel is a **busbar** (copper busbar, per metre).
sources: ['D2']
Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
answer:  The price of one busbar is 95.0.
sources: []


In [33]:
# Cell 31: Check that each run reports its sources and unsupported numbers
import importlib
import sys

for name in ["estimation_assistant.agent", "estimation_assistant.checks", "estimation_assistant.schemas"]:
    sys.modules.pop(name, None)
agent = importlib.import_module("estimation_assistant.agent")

for question in [
    "What is the price of one busbar?",
    "How much do 3 busbars cost in total?",
    "Which part carries current between breakers in a panel?",
]:
    result = agent.run_agent(question)
    print("answer:     ", result.answer)
    print("sources:    ", result.sources)
    print("unsupported:", result.unsupported)
    print()

Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
answer:      The price of one busbar is 95.0.
sources:     []
unsupported: []

Round 1
  lookup_price({'part_name': 'busbar'}) -> 95.0
Round 2
  line_total({'quantity': 3, 'unit_price': 95}) -> 285
Round 3
answer:      The total cost for 3 busbars is 285.00 pounds.
sources:     []
unsupported: []

Round 1
  search_notes({'question': 'which part carries current between breakers in a panel'}) -> Source D2 (score 0.861): A busbar carries current between circuit breakers in a panel.
Round 2
answer:      The part that carries current between breakers in a panel is a **busbar**.
sources:     ['D2']
unsupported: []



In [34]:
# Cell 32: Confirm the package files exist on this Colab session
!ls -l ai_engine/estimation_assistant

total 24
-rw-r--r-- 1 root root 3221 Nov  3 14:34 agent.py
-rw-r--r-- 1 root root 1307 Nov  3 14:34 checks.py
-rw-r--r-- 1 root root    0 Nov  3 14:34 __init__.py
drwxr-xr-x 2 root root 4096 Nov  3 14:34 __pycache__
-rw-r--r-- 1 root root 1909 Nov  3 14:34 retrieval.py
-rw-r--r-- 1 root root  245 Nov  3 14:34 schemas.py
-rw-r--r-- 1 root root 1333 Nov  3 14:34 tools.py
